# 07 — Analisis Domain Shift (Kontrol dan Domain Classifier)

Notebook ini menghitung ulang angka yang dipakai pada Subbab 3.4 naskah, **dari fitur dan citra yang sama dengan notebook 03–06**, supaya setiap angka bisa ditelusuri ke output notebook.

Isi:
1. Cek bahwa baseline S3/S4 terulang persis (MobileNetV2 dan EfficientNetB0).
2. Kontrol ukuran data dan keseimbangan kelas, A→B (kedua model, 10 pengulangan).
3. Domain classifier (regresi logistik) dan AUC, kedua model.
4. Kontrol warna dan pencahayaan pada EfficientNetB0 (ekstraksi fitur ulang dari citra), dengan pembanding (reference) dari pipeline yang sama.

Semua hasil disimpan ke `baseline_v2/multi_regime/domain_shift_analysis/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json, numpy as np, pandas as pd, time
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

ROOT = Path('/content/drive/MyDrive/eksperimen_baru')
BASE = ROOT / 'baseline_v2'
OUT = BASE / 'multi_regime' / 'domain_shift_analysis'
OUT.mkdir(parents=True, exist_ok=True)

CONFIG = json.load(open(BASE / 'config/config.json'))
CLASS_WEIGHT = CONFIG['svm']['class_weight']
MAX_ITER = CONFIG['svm']['max_iter']
TOL = CONFIG['svm']['tolerance']
RANDOM_STATE = CONFIG['random_state']
print('class_weight', CLASS_WEIGHT, '| max_iter', MAX_ITER, '| tol', TOL, '| random_state', RANDOM_STATE)

def build_svm(C):
    return Pipeline([('scaler', StandardScaler()),
                     ('svm', LinearSVC(C=C, class_weight=CLASS_WEIGHT, max_iter=MAX_ITER,
                                       tol=TOL, dual='auto', random_state=RANDOM_STATE))])

# C terpilih (best_c.csv masing-masing model); sumber = dataset sumber
MODELS = {
    'mobilenetv2_frozen':    {'root': BASE/'features/frozen/mobilenetv2_frozen',    'C': {'a': 0.001, 'b': 0.001}},
    'efficientnetb0_frozen': {'root': BASE/'features/frozen/efficientnetb0_frozen', 'C': {'a': 0.01,  'b': 0.001}},
}
EXPECTED = {('mobilenetv2_frozen','A_to_B'):0.3561, ('mobilenetv2_frozen','B_to_A'):0.2657,
            ('efficientnetb0_frozen','A_to_B'):0.3262, ('efficientnetb0_frozen','B_to_A'):0.3087}

def load(model, split):
    r = MODELS[model]['root']
    return (np.asarray(np.load(r/f'{split}_features.npy')), np.load(r/f'{split}_labels.npy'),
            pd.read_csv(r/f'{split}_index.csv'))

def macro_f1(y, p): return f1_score(y, p, average='macro', zero_division=0)

Mounted at /content/drive
class_weight balanced | max_iter 50000 | tol 0.0001 | random_state 42


## 1. Cek baseline (harus sama dengan Tabel 5 naskah)

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# 1) Muat fitur
FEAT = {}
for m in MODELS:
    FEAT[m] = {s: load(m, s) for s in ['train_a','val_a','test_a','train_b','val_b','test_b',
                                       'external_a_full','external_b_full']}

def pool(m, regime):
    d = FEAT[m]
    return (np.vstack([d[f'train_{regime}'][0], d[f'val_{regime}'][0]]),
            np.concatenate([d[f'train_{regime}'][1], d[f'val_{regime}'][1]]))

# 2) Angka Tabel 5 di jurnal: (accuracy, precision, recall, macro F1)
TABEL5 = {
    ('S3 (A→B)', 'mobilenetv2_frozen'):    (0.3667, 0.3497, 0.3667, 0.3561),
    ('S3 (A→B)', 'efficientnetb0_frozen'): (0.3583, 0.4095, 0.3583, 0.3262),
    ('S4 (B→A)', 'mobilenetv2_frozen'):    (0.2701, 0.3693, 0.3238, 0.2657),
    ('S4 (B→A)', 'efficientnetb0_frozen'): (0.3367, 0.3440, 0.3277, 0.3087),
}
NAMA = {'mobilenetv2_frozen': 'MobileNetV2-SVM', 'efficientnetb0_frozen': 'EfficientNetB0-SVM'}

# 3) Hitung baseline S3 dan S4
rows = []
for skenario, src, tgt in [('S3 (A→B)', 'a', 'external_b_full'), ('S4 (B→A)', 'b', 'external_a_full')]:
    for m in MODELS:
        X, y = pool(m, src)
        svm = build_svm(MODELS[m]['C'][src]).fit(X, y)
        Xt, yt, _ = FEAT[m][tgt]
        pred = svm.predict(Xt)
        prec, rec, f1, _ = precision_recall_fscore_support(yt, pred, average='macro', zero_division=0)
        acc = accuracy_score(yt, pred)
        rows.append({'Scenarios': skenario, 'Model': NAMA[m],
                     'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'Macro F1': f1,
                     '_key': (skenario, m)})

raw = pd.DataFrame(rows)

# 4) Tabel dibulatkan 4 digit (format yang ditulis di jurnal)
tabel = raw.drop(columns='_key').copy()
for c in ['Accuracy', 'Precision', 'Recall', 'Macro F1']:
    tabel[c] = tabel[c].round(4)
display(tabel)

# 5) Cek terhadap Tabel 5 jurnal
ok = True
for _, r in raw.iterrows():
    target = TABEL5[r['_key']]
    got = tuple(round(r[c], 4) for c in ['Accuracy', 'Precision', 'Recall', 'Macro F1'])
    sama = got == target
    ok &= sama
    print(f"{r['Scenarios']} | {r['Model']:<19} | notebook {got} | jurnal {target} | {'SAMA' if sama else 'BEDA'}")
assert ok, 'Ada angka yang beda dari Tabel 5, cek baris yang BEDA di atas.'
print('Baseline OK: semua angka Tabel 5 terulang')

#

,Scenarios,Model,Accuracy,Precision,Recall,Macro F1
0,S3 (A→B),MobileNetV2-SVM,0.3667,0.3497,0.3667,0.3561
1,S3 (A→B),EfficientNetB0-SVM,0.3583,0.4095,0.3583,0.3262
2,S4 (B→A),MobileNetV2-SVM,0.2701,0.3693,0.3238,0.2657
3,S4 (B→A),EfficientNetB0-SVM,0.3367,0.3440,0.3277,0.3087


S3 (A→B) | MobileNetV2-SVM     | notebook (0.3667, 0.3497, 0.3667, 0.3561) | jurnal (0.3667, 0.3497, 0.3667, 0.3561) | SAMA
S3 (A→B) | EfficientNetB0-SVM  | notebook (0.3583, 0.4095, 0.3583, 0.3262) | jurnal (0.3583, 0.4095, 0.3583, 0.3262) | SAMA
S4 (B→A) | MobileNetV2-SVM     | notebook (0.2701, 0.3693, 0.3238, 0.2657) | jurnal (0.2701, 0.3693, 0.3238, 0.2657) | SAMA
S4 (B→A) | EfficientNetB0-SVM  | notebook (0.3367, 0.344, 0.3277, 0.3087) | jurnal (0.3367, 0.344, 0.3277, 0.3087) | SAMA
Baseline OK: semua angka Tabel 5 terulang


## 2. Kontrol ukuran data dan keseimbangan kelas (A→B)

In [ ]:
N_SEEDS = 10
def per_class_sample(y, n, rng):
    idx = []
    for c in np.unique(y):
        ci = np.where(y == c)[0]
        idx += rng.choice(ci, size=n, replace=False).tolist()
    return np.array(idx)

ctrl_rows = []
for m in MODELS:
    XA, yA = pool(m, 'a'); XB, yB = pool(m, 'b')
    n_small = len(yB) // 3                       # samakan ukuran dengan pool B (216 -> 72/kelas)
    n_bal = int(np.bincount(yA).min())           # seimbang: jumlah kelas terkecil di pool A
    print(m, '| per-kelas ukuran-B:', n_small, '| per-kelas seimbang:', n_bal, '| hitungan kelas pool A:', np.bincount(yA).tolist())
    Xt, yt, _ = FEAT[m]['external_b_full']
    full = macro_f1(yt, build_svm(MODELS[m]['C']['a']).fit(XA, yA).predict(Xt))
    ctrl_rows.append({'model': m, 'condition': 'full (baseline)', 'mean': full, 'sd': np.nan, 'n_runs': 1})
    for cond, n in [(f'size-matched {n_small}/kelas', n_small), (f'class-balanced {n_bal}/kelas', n_bal)]:
        sc = []
        for seed in range(N_SEEDS):
            idx = per_class_sample(yA, n, np.random.default_rng(seed))
            sc.append(macro_f1(yt, build_svm(MODELS[m]['C']['a']).fit(XA[idx], yA[idx]).predict(Xt)))
        ctrl_rows.append({'model': m, 'condition': cond, 'mean': np.mean(sc), 'sd': np.std(sc, ddof=1), 'n_runs': N_SEEDS})
ctrl_df = pd.DataFrame(ctrl_rows); display(ctrl_df.round(4))
ctrl_df.to_csv(OUT/'control_size_balance_both_models.csv', index=False)

mobilenetv2_frozen | per-kelas ukuran-B: 72 | per-kelas seimbang: 421 | hitungan kelas pool A: [421, 1554, 972]
efficientnetb0_frozen | per-kelas ukuran-B: 72 | per-kelas seimbang: 421 | hitungan kelas pool A: [421, 1554, 972]


,model,condition,mean,sd,n_runs
0,mobilenetv2_frozen,full (baseline),0.3561,NaN,1
1,mobilenetv2_frozen,size-matched 72/kelas,0.2980,0.0585,10
2,mobilenetv2_frozen,class-balanced 421/kelas,0.2852,0.0381,10
3,efficientnetb0_frozen,full (baseline),0.3262,NaN,1
4,efficientnetb0_frozen,size-matched 72/kelas,0.2404,0.0418,10
5,efficientnetb0_frozen,class-balanced 421/kelas,0.2723,0.0334,10


## 3. Domain classifier (regresi logistik) dan AUC

In [ ]:
auc_rows = []
for m in MODELS:
    XA, yA, _ = FEAT[m]['external_a_full']; XB, yB, _ = FEAT[m]['external_b_full']
    aucs = []
    for seed in [42, 52, 62, 72, 82]:
        rng = np.random.default_rng(seed)
        ia = per_class_sample(yA, 80, rng)                      # 240 citra A (80 per kelas), sama jumlahnya dengan B
        X = np.vstack([XA[ia], XB]); d = np.r_[np.zeros(len(ia)), np.ones(len(XB))]
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(X, d):
            clf = Pipeline([('s', StandardScaler()), ('lr', LogisticRegression(C=0.01, max_iter=5000))]).fit(X[tr], d[tr])
            aucs.append(roc_auc_score(d[te], clf.decision_function(X[te])))
    auc_rows.append({'model': m, 'auc_mean': np.mean(aucs), 'auc_sd': np.std(aucs, ddof=1), 'n_folds': len(aucs)})
auc_df = pd.DataFrame(auc_rows); display(auc_df.round(4)); auc_df.to_csv(OUT/'domain_classifier_auc.csv', index=False)

,model,auc_mean,auc_sd,n_folds
0,mobilenetv2_frozen,1.0,0.0,25
1,efficientnetb0_frozen,1.0,0.0,25


## 4. Kontrol warna dan pencahayaan (EfficientNetB0)

Fitur diekstraksi ulang dari citra untuk setiap perlakuan. Langkah pertama **memeriksa bahwa pipeline ekstraksi ini menghasilkan fitur yang sama dengan fitur tersimpan** (tanpa perlakuan). Ini yang menjelaskan atau menghilangkan selisih 0.3082 vs 0.3262. Interpolasi resize dipilih otomatis yang paling cocok dengan fitur tersimpan.

Perlakuan (semua dijalankan pada citra sumber dan target, SVM dilatih ulang dengan C yang sama):
- `reference`: tanpa perubahan
- `grayscale`
- `per_image_norm`: tiap citra dinormalkan per kanal ke rata-rata 127.5 dan simpangan baku 50
- `dataset_color_match`: statistik kanal tiap dataset diselaraskan ke statistik Dataset A (tanpa label)
- `center_crop_width50`: hanya bagian tengah 50% lebar citra (tinggi penuh) lalu resize 224×224

In [ ]:
import tensorflow as tf
from PIL import Image
from tensorflow.keras.applications.efficientnet import EfficientNetB0, preprocess_input
M = 'efficientnetb0_frozen'
Xa_s, ya, idxa = FEAT[M]['external_a_full']; Xb_s, yb, idxb = FEAT[M]['external_b_full']
paths_a = idxa['path'].tolist(); paths_b = idxb['path'].tolist()
extractor = EfficientNetB0(weights='imagenet', include_top=False, pooling='avg', input_shape=(224,224,3))
RES = {'nearest': Image.NEAREST, 'bilinear': Image.BILINEAR, 'bicubic': Image.BICUBIC, 'lanczos': Image.LANCZOS}

def load_u8(path, resample, crop_w=None):
    im = Image.open(path).convert('RGB')
    if crop_w:
        w, h = im.size; cw = int(w*crop_w); l = (w-cw)//2; im = im.crop((l, 0, l+cw, h))
    return np.asarray(im.resize((224,224), resample), dtype=np.uint8)

def extract(arrs_u8, bs=64):
    out = []
    for i in range(0, len(arrs_u8), bs):
        b = preprocess_input(np.asarray(arrs_u8[i:i+bs], dtype=np.float32))
        out.append(extractor.predict(b, verbose=0))
    return np.vstack(out)

def cos(a, b): return (np.sum(a*b, 1) / (np.linalg.norm(a,axis=1)*np.linalg.norm(b,axis=1))).mean()

# cek kecocokan dengan fitur tersimpan (30 citra A + 30 citra B)
chk = {}
for name, rs in RES.items():
    ua = [load_u8(p, rs) for p in paths_a[:30]]; ub = [load_u8(p, rs) for p in paths_b[:30]]
    chk[name] = (cos(extract(ua), Xa_s[:30]) + cos(extract(ub), Xb_s[:30])) / 2
print('kemiripan kosinus dengan fitur tersimpan:', {k: round(v,5) for k,v in chk.items()})
BEST_RES = max(chk, key=chk.get); print('dipakai:', BEST_RES)
assert chk[BEST_RES] > 0.99, 'Pipeline ekstraksi belum cocok dengan fitur tersimpan (cek preprocessing di notebook ekstraksi fitur).'

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
kemiripan kosinus dengan fitur tersimpan: {'nearest': np.float32(0.88685), 'bilinear': np.float32(0.99967), 'bicubic': np.float32(0.98976), 'lanczos': np.float32(0.97109)}
dipakai: bilinear


In [ ]:
t0 = time.time()
U_a = np.stack([load_u8(p, RES[BEST_RES]) for p in paths_a]); U_b = np.stack([load_u8(p, RES[BEST_RES]) for p in paths_b])
print('citra dimuat', U_a.shape, U_b.shape, round(time.time()-t0), 's')
mu_a, sd_a = U_a.reshape(-1,3).mean(0), U_a.reshape(-1,3).std(0)
mu_b, sd_b = U_b.reshape(-1,3).mean(0), U_b.reshape(-1,3).std(0)

def t_gray(U):
    g = (U.astype(np.float32) @ np.array([0.299,0.587,0.114], dtype=np.float32))[..., None]
    return np.repeat(g, 3, axis=-1)
def t_perimg(U):
    x = U.astype(np.float32); m = x.mean((1,2), keepdims=True); s = x.std((1,2), keepdims=True) + 1e-6
    return np.clip((x - m) / s * 50 + 127.5, 0, 255)
def t_match(U, mu, sd):
    return np.clip((U.astype(np.float32) - mu) / sd * sd_a + mu_a, 0, 255)

TREAT = {
  'reference':           (lambda U, d: U, None),
  'grayscale':           (lambda U, d: t_gray(U), None),
  'per_image_norm':      (lambda U, d: t_perimg(U), None),
  'dataset_color_match': (lambda U, d: t_match(U, *( (mu_a, sd_a) if d == 'A' else (mu_b, sd_b) )), None),
}
FE = {}
for name, (fn, _) in TREAT.items():
    FE[name] = (np.vstack([extract(fn(U_a[i:i+256], 'A')) for i in range(0, len(U_a), 256)]),
                extract(fn(U_b, 'B')))
    print(name, 'selesai', round(time.time()-t0), 's')
# center crop: dari file asli
ca = np.stack([load_u8(p, RES[BEST_RES], crop_w=0.5) for p in paths_a]); cb = np.stack([load_u8(p, RES[BEST_RES], crop_w=0.5) for p in paths_b])
FE['center_crop_width50'] = (extract(ca), extract(cb)); print('center_crop selesai', round(time.time()-t0), 's')
np.savez_compressed(OUT/'photometric_features_efficientnetb0.npz', **{f'{k}_{i}': v[i] for k, v in FE.items() for i in (0,1)})

citra dimuat (3276, 224, 224, 3) (240, 224, 224, 3) 1546 s
reference selesai 1601 s
grayscale selesai 1617 s
per_image_norm selesai 1650 s
dataset_color_match selesai 1677 s
center_crop selesai 1721 s


In [ ]:
row_a = {p: i for i, p in enumerate(paths_a)}; row_b = {p: i for i, p in enumerate(paths_b)}
def pool_rows(regime, rowmap):
    ii = []
    for s in (f'train_{regime}', f'val_{regime}'):
        ii += [rowmap[p] for p in FEAT[M][s][2]['path']]
    return np.array(ii)
pa = pool_rows('a', row_a); pb = pool_rows('b', row_b)
ta = np.array([row_a[p] for p in FEAT[M]['test_a'][2]['path']])
CA, CB = MODELS[M]['C']['a'], MODELS[M]['C']['b']
res = []
for name, (Fa, Fb) in FE.items():
    f_ab = macro_f1(yb, build_svm(CA).fit(Fa[pa], ya[pa]).predict(Fb))
    f_ba = macro_f1(ya, build_svm(CB).fit(Fb[pb], yb[pb]).predict(Fa))
    f_aa = macro_f1(ya[ta], build_svm(CA).fit(Fa[pa], ya[pa]).predict(Fa[ta]))
    aucs = []
    for seed in [42, 52, 62, 72, 82]:
        ia = per_class_sample(ya, 80, np.random.default_rng(seed))
        X = np.vstack([Fa[ia], Fb]); d = np.r_[np.zeros(len(ia)), np.ones(len(Fb))]
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(X, d):
            clf = Pipeline([('s', StandardScaler()), ('lr', LogisticRegression(C=0.01, max_iter=5000))]).fit(X[tr], d[tr])
            aucs.append(roc_auc_score(d[te], clf.decision_function(X[te])))
    res.append({'treatment': name, 'A_to_B': f_ab, 'B_to_A': f_ba, 'A_to_A': f_aa, 'domain_AUC': np.mean(aucs)})
photo_df = pd.DataFrame(res); display(photo_df.round(4)); photo_df.to_csv(OUT/'photometric_controls_efficientnetb0.csv', index=False)
print('Baseline Tabel 5 (fitur tersimpan): A_to_B 0.3262 | B_to_A 0.3087')

,treatment,A_to_B,B_to_A,A_to_A,domain_AUC
0,reference,0.3082,0.3049,0.9266,1.0
1,grayscale,0.2581,0.1901,0.9055,1.0
2,per_image_norm,0.1974,0.3589,0.9081,1.0
3,dataset_color_match,0.2417,0.3974,0.9266,1.0
4,center_crop_width50,0.3323,0.1794,0.9126,1.0


Baseline Tabel 5 (fitur tersimpan): A_to_B 0.3262 | B_to_A 0.3087


In [ ]:
from sklearn.utils import resample

def match_feats(U, mu_f, sd_f, mu_t, sd_t):
    f = lambda Z: np.clip((Z.astype(np.float32) - mu_f) / sd_f * sd_t + mu_t, 0, 255)
    return np.vstack([extract(f(U[i:i+256])) for i in range(0, len(U), 256)])

Fa_as_b = match_feats(U_a, mu_a, sd_a, mu_b, sd_b)   # A dengan statistik warna B
Fb_as_a = match_feats(U_b, mu_b, sd_b, mu_a, sd_a)   # B dengan statistik warna A
Fa_ref, Fb_ref = FE['reference']

def run(Xtr, ytr, Xte, yte, C, n_boot=1000):
    pred = build_svm(C).fit(Xtr, ytr).predict(Xte)
    f1 = macro_f1(yte, pred); idx = np.arange(len(yte)); sc = []
    for i in range(n_boot):
        s = resample(idx, replace=True, n_samples=len(idx), stratify=yte, random_state=RANDOM_STATE + i)
        sc.append(macro_f1(yte[s], pred[s]))
    lo, hi = np.percentile(sc, [2.5, 97.5])
    return f1, lo, hi

conds = {}
for name, (Fa, Fb) in FE.items():
    conds[name] = {'A_to_B': (Fa[pa], ya[pa], Fb, yb, CA), 'B_to_A': (Fb[pb], yb[pb], Fa, ya, CB)}
conds['match_source_to_target'] = {'A_to_B': (Fa_as_b[pa], ya[pa], Fb_ref, yb, CA),
                                   'B_to_A': (Fb_as_a[pb], yb[pb], Fa_ref, ya, CB)}
conds['match_target_to_source'] = {'A_to_B': (Fa_ref[pa], ya[pa], Fb_as_a, yb, CA),
                                   'B_to_A': (Fb_ref[pb], yb[pb], Fa_as_b, ya, CB)}

rows = []
for name, d in conds.items():
    r = {'treatment': name}
    for arah, args in d.items():
        f1, lo, hi = run(*args)
        r[arah] = round(f1, 4); r[arah + '_CI95'] = f'[{lo:.3f}, {hi:.3f}]'
    rows.append(r)
final_df = pd.DataFrame(rows)
pd.set_option('display.width', 220)
display(final_df)
final_df.to_csv(OUT / 'photometric_controls_with_ci.csv', index=False)

,treatment,A_to_B,A_to_B_CI95,B_to_A,B_to_A_CI95
0,reference,0.3082,"[0.256, 0.358]",0.3049,"[0.290, 0.321]"
1,grayscale,0.2581,"[0.209, 0.312]",0.1901,"[0.180, 0.201]"
2,per_image_norm,0.1974,"[0.163, 0.235]",0.3589,"[0.345, 0.373]"
3,dataset_color_match,0.2417,"[0.187, 0.297]",0.3974,"[0.381, 0.413]"
4,center_crop_width50,0.3323,"[0.285, 0.385]",0.1794,"[0.171, 0.188]"
5,match_source_to_target,0.1907,"[0.151, 0.235]",0.3974,"[0.381, 0.413]"
6,match_target_to_source,0.2417,"[0.187, 0.297]",0.2319,"[0.220, 0.244]"


## 5. Cara memakai hasil
- Salin angka dari tabel di atas ke Subbab 3.4 (kontrol ukuran data dan keseimbangan kelas untuk **kedua model**, AUC, dan kontrol warna/pencahayaan).
- Kalau `reference` ≈ baseline Tabel 5 (0.3262 dan 0.3087), selisih lama (0.3082 dan 0.3049) hilang dan kalimat di 3.4 cukup memakai angka baru.
- Kalau `reference` berbeda dari baseline, tulis satu kalimat: kontrol dijalankan dengan ekstraksi fitur ulang dari citra, sehingga nilai referensinya sedikit berbeda dari Tabel 5.
- Kalau arah efek berbeda dari teks lama (misalnya penyesuaian warna tidak lagi menaikkan A→B), **teks dan kesimpulan yang mengikuti angka baru**, bukan sebaliknya.